# Day 52 — Attention from scratch, and the four things everyone changes about it

Attention is a weighted average. The weights come from a softmax over similarity
scores, and almost everything that is hard about it — overflow, the `sqrt(d)`, the
mask, the memory bill, FlashAttention, paged KV, MQA/GQA/MLA, linear attention — is
about how you compute that average without building the thing it is an average over.

In [1]:
import math

import numpy as np

# ================================================================ 1. softmax
def softmax_naive(x, axis=-1):
    """The formula as written.  exp() of a large number is inf, and inf / inf is nan."""
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def softmax(x, axis=-1):
    """Subtract the row max first.

    softmax(x) == softmax(x - c) for any constant c, because the c cancels between
    numerator and denominator.  Choosing c = max(x) makes the largest exponent exactly
    exp(0) = 1, so nothing can overflow and the largest term is never lost.
    """
    x = np.asarray(x, dtype=float)
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)


# ================================================================ 2. attention
def attention(Q, K, V, mask=None, scale=None):
    """softmax(QK^T / sqrt(d)) V - four lines, and the rest of the file is about
    what each line costs.

    Q is (n, d), K and V are (m, d).  The score matrix is (n, m): one number for
    every pair of positions, which is where the quadratic memory comes from.
    """
    d = Q.shape[-1]
    scores = Q @ K.T / (scale if scale is not None else math.sqrt(d))
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    w = softmax(scores, axis=-1)
    return w @ V, w


def causal_mask(n, m=None):
    """True where query row i is allowed to look at key j.

    For the square case this is j <= i.  When there are more keys than queries (the
    decode case, where the queries are the newest m - n positions) the diagonal is
    shifted by m - n, so row i may look at j <= i + m - n.
    """
    m = n if m is None else m
    return np.tril(np.ones((n, m), dtype=bool), k=m - n)


rng = np.random.default_rng(0)
Q, K, V = rng.standard_normal((4, 8)), rng.standard_normal((6, 8)), rng.standard_normal((6, 3))
out, w = attention(Q, K, V)
print('weights', w.shape, 'row sums', np.round(w.sum(axis=-1), 6))
print('row 0  ', np.round(w[0], 4))
print()
print('V   range', np.round(V.min(axis=0), 3), 'to', np.round(V.max(axis=0), 3))
print('out range', np.round(out.min(axis=0), 3), 'to', np.round(out.max(axis=0), 3))
print('-> every output is a convex combination of the values; attention cannot invent one')

weights (4, 6) row sums [1. 1. 1. 1.]
row 0   [0.1902 0.2974 0.0508 0.0912 0.0932 0.2772]

V   range [-1.091 -1.278 -0.755] to [1.689 1.295 1.574]
out range [ 0.133 -0.485 -0.283] to [0.701 0.566 0.103]
-> every output is a convex combination of the values; attention cannot invent one


## Silent failure #1: `exp()` of a large number

`softmax(x) == softmax(x - c)` for any constant `c`, because `c` cancels between the
numerator and the denominator. Taking `c = max(x)` makes the largest exponent
`exp(0) = 1`, so nothing can overflow. In fp16 a logit of 12 is already enough.

In [2]:
def overflow_threshold(dtype=np.float32):
    """The smallest integer whose exp() is inf in this dtype."""
    info = np.finfo(dtype)
    k = 1
    with np.errstate(over='ignore'):
        while np.isfinite(np.exp(np.array(k, dtype=dtype))):
            k += 1
    return k, float(info.max)


for dt in (np.float32, np.float16):
    k, mx = overflow_threshold(dt)
    print(f'{np.dtype(dt).name}: holds up to {mx:.3e}, exp({k}) is already inf')
print()
big = np.array([[1000.0, 1001.0, 999.0]])
with np.errstate(over='ignore', invalid='ignore'):
    print('softmax_naive:', softmax_naive(big))
print('softmax      :', np.round(softmax(big), 6))

float32: holds up to 3.403e+38, exp(89) is already inf
float16: holds up to 6.550e+04, exp(12) is already inf

softmax_naive: [[nan nan nan]]
softmax      : [[0.244728 0.665241 0.090031]]


## Why the 1 / sqrt(d)

A score is a dot product of `d` terms, so its standard deviation grows like `sqrt(d)`.
Without the scaling, a wide head puts almost all of its weight on one key — an argmax
with almost no gradient reaching the others.

In [3]:
print(f'{"d":>6}{"std of raw scores":>20}{"sqrt(d)":>10}{"max weight raw":>16}{"scaled":>9}')
for dd in (8, 64, 512, 4096):
    q, kk = rng.standard_normal((256, dd)), rng.standard_normal((256, dd))
    raw = q @ kk.T
    print(f'{dd:>6}{np.std(raw):>20.3f}{math.sqrt(dd):>10.2f}'
          f'{softmax(raw, axis=-1).max(axis=-1).mean():>16.4f}'
          f'{softmax(raw / math.sqrt(dd), axis=-1).max(axis=-1).mean():>9.4f}')

     d   std of raw scores   sqrt(d)  max weight raw   scaled
     8               2.822      2.83          0.3070   0.0430
    64               7.999      8.00          0.7484   0.0457
   512              22.689     22.63          0.9073   0.0415
  4096              64.038     64.00          0.9678   0.0453


## Silent failure #2: masking after the softmax instead of before

Put `-inf` in the scores and the future gets exactly zero weight while the row still
sums to 1. Zero it *after* the softmax and every early position is quietly scaled down
by however much future it was not allowed to see.

In [4]:
def attention_mask_after_softmax(Q, K, V, mask):
    """The bug: softmax first, then zero the future.

    The rows still sum to 1 before the zeroing, so afterwards they do not, and every
    position is scaled down by however much future it was not allowed to see.  No
    exception, no nan - just a quietly wrong weighted average.
    """
    d = Q.shape[-1]
    w = softmax(Q @ K.T / math.sqrt(d), axis=-1)
    w = np.where(mask, w, 0.0)
    return w @ V, w


nn = 5
Qc, Kc, Vc = rng.standard_normal((nn, 8)), rng.standard_normal((nn, 8)), rng.standard_normal((nn, 3))
msk = causal_mask(nn)
_, w_ok = attention(Qc, Kc, Vc, mask=msk)
_, w_bad = attention_mask_after_softmax(Qc, Kc, Vc, msk)
print('masked before softmax, row sums:', np.round(w_ok.sum(axis=-1), 6))
print('masked after  softmax, row sums:', np.round(w_bad.sum(axis=-1), 6))

masked before softmax, row sums: [1. 1. 1. 1. 1.]
masked after  softmax, row sums: [0.174385 0.313089 0.675729 0.651268 1.      ]


## FlashAttention: the same number, one pass, without the matrix

Walk the keys in blocks carrying a running max and a running sum, and rescale what you
already have by `exp(old_max - new_max)` whenever the max moves. It is an exact rewrite
of the same sum, which is why FlashAttention changes the memory and the speed and not
the model.

In [5]:
# ================================================================ 3. online softmax
def attention_online(Q, K, V, block=32, mask=None):
    """FlashAttention's identity, written plainly: one pass over K and V in blocks,
    carrying a running max and a running sum, and rescaling what you already have
    whenever the max moves.

    The n x m score matrix is never materialised - only one block of it at a time -
    and the result is the same number, not an approximation.
    """
    n, d = Q.shape
    m = K.shape[0]
    out = np.zeros((n, V.shape[1]))
    run_max = np.full(n, -np.inf)
    run_sum = np.zeros(n)
    for s in range(0, m, block):
        e = min(s + block, m)
        s_blk = Q @ K[s:e].T / math.sqrt(d)
        if mask is not None:
            s_blk = np.where(mask[:, s:e], s_blk, -np.inf)
        blk_max = np.max(s_blk, axis=-1)
        new_max = np.maximum(run_max, blk_max)
        new_max = np.where(np.isfinite(new_max), new_max, 0.0)
        # rescale what we already accumulated to the new maximum
        alpha = np.exp(run_max - new_max)
        p = np.exp(s_blk - new_max[:, None])
        p = np.where(np.isfinite(p), p, 0.0)
        run_sum = run_sum * alpha + p.sum(axis=-1)
        out = out * alpha[:, None] + p @ V[s:e]
        run_max = new_max
    return out / np.where(run_sum == 0, 1.0, run_sum)[:, None]


Qf, Kf, Vf = (rng.standard_normal((64, 64)), rng.standard_normal((256, 64)),
              rng.standard_normal((256, 32)))
ref, _ = attention(Qf, Kf, Vf)
for blk in (1, 8, 32, 128, 256):
    print(f'block {blk:>4}: max abs difference {np.abs(attention_online(Qf, Kf, Vf, block=blk) - ref).max():.3e}')

block    1: max abs difference 1.221e-15
block    8: max abs difference 6.661e-16
block   32: max abs difference 6.661e-16
block  128: max abs difference 4.996e-16
block  256: max abs difference 5.274e-16


## The bill the quadratic term sends, and the two ways to lower it

The score matrix has one entry per pair of positions, per head, per layer. The KV cache
has one K and one V per token — unless you change what has to be cached at all.

In [6]:
# ================================================================ 4. the memory bill
def score_matrix_bytes(n, heads, bytes_per_elem=2):
    """How big the n x n score matrix is, per layer, if you build it."""
    return n * n * heads * bytes_per_elem


def kv_cache_bytes_per_token(layers=32, heads=32, head_dim=128, kv_heads=None,
                             latent=None, rope_dim=0, bytes_per_elem=2):
    """Bytes of KV cache one token costs, for the four common layouts.

    MHA   : kv_heads == heads          - one K and one V per head
    MQA   : kv_heads == 1              - all heads share one K and one V
    GQA   : 1 < kv_heads < heads       - groups of heads share
    MLA   : latent is not None         - cache one low-rank vector per token instead
                                         of K and V, plus the part that carries RoPE
    """
    if latent is not None:
        return layers * (latent + rope_dim) * bytes_per_elem
    kv_heads = heads if kv_heads is None else kv_heads
    return layers * 2 * kv_heads * head_dim * bytes_per_elem


def linear_state_bytes(layers=32, heads=32, head_dim=128, bytes_per_elem=2):
    """A delta-rule / linear-attention layer carries a head_dim x head_dim matrix per
    head instead of a growing cache.  It does not depend on the sequence length."""
    return layers * heads * head_dim * head_dim * bytes_per_elem


print(f'{"context":>10}{"score matrix / layer":>24}')
for ctx in (1024, 4096, 16384, 131072):
    print(f'{ctx:>10,}{score_matrix_bytes(ctx, 32) / 2**30:>21.2f} GB')
print()
L, H, D = 32, 32, 128
rows = [('MHA', kv_cache_bytes_per_token(L, H, D)),
        ('GQA', kv_cache_bytes_per_token(L, H, D, kv_heads=8)),
        ('MLA', kv_cache_bytes_per_token(L, H, D, latent=512, rope_dim=64)),
        ('MQA', kv_cache_bytes_per_token(L, H, D, kv_heads=1))]
print(f'{"layout":>8}{"bytes/token":>13}{"128k context":>15}{"vs MHA":>9}')
for nm, b in rows:
    print(f'{nm:>8}{b:>13,}{131072 * b / 2**30:>12.1f} GB{rows[0][1] / b:>8.1f}x')
print()
print(f'a delta-rule state is {linear_state_bytes(L, H, D) / 2**20:.0f} MB and does not depend on the context')

   context    score matrix / layer
     1,024                 0.06 GB
     4,096                 1.00 GB
    16,384                16.00 GB
   131,072              1024.00 GB

  layout  bytes/token   128k context   vs MHA
     MHA      524,288        64.0 GB     1.0x
     GQA      131,072        16.0 GB     4.0x
     MLA       36,864         4.5 GB    14.2x
     MQA       16,384         2.0 GB    32.0x

a delta-rule state is 32 MB and does not depend on the context


## Paged attention: the KV cache is a memory allocator problem

A contiguous cache has to reserve for the longest the sequence might get. Pages are
handed out as it grows, so the only waste left is the tail of the last page.

In [7]:
# ================================================================ 6. paged KV
def contiguous_waste(lengths, capacity):
    """Reserve `capacity` slots per sequence up front, the way a contiguous cache must.

    Every sequence is allocated for the longest it might get, so the waste is the gap
    between what was reserved and what was used.
    """
    reserved = len(lengths) * capacity
    used = sum(lengths)
    return reserved, used, (reserved - used) / reserved


def paged_waste(lengths, page_size):
    """Hand out fixed-size pages on demand instead.

    The only waste left is the unused tail of each sequence's last page, which is at
    most page_size - 1 slots per sequence no matter how long the sequence is.
    """
    pages = sum(math.ceil(n / page_size) for n in lengths)
    reserved = pages * page_size
    used = sum(lengths)
    return pages, reserved, used, (reserved - used) / reserved


lengths = [12, 300, 48, 900, 7, 2048, 64, 130]
reserved, used, waste = contiguous_waste(lengths, 4096)
print(f'one block per sequence: {reserved:,} slots, {used:,} used, {100 * waste:.1f}% wasted')
for page in (16, 64, 256):
    pages, res, usd, w = paged_waste(lengths, page)
    print(f'pages of {page:>4}:        {res:,} slots, {usd:,} used, {100 * w:.1f}% wasted ({pages} pages)')

one block per sequence: 32,768 slots, 3,509 used, 89.3% wasted
pages of   16:        3,552 slots, 3,509 used, 1.2% wasted (222 pages)
pages of   64:        3,776 slots, 3,509 used, 7.1% wasted (59 pages)
pages of  256:        4,864 slots, 3,509 used, 27.9% wasted (19 pages)


## The delta rule: a state you write to, not append to

`S <- S (I - beta k kᵀ) + beta v kᵀ`. The first term erases whatever `S` currently
associates with `k`, the second writes `v` there instead; a real layer reads with its
own query projection, `o = S @ q`. Gated variants add a forget factor so old
associations decay: Gated DeltaNet one scalar per head, Kimi's KDA one per channel.
The state is still a fixed-size matrix.

In [8]:
# ================================================================ 5. linear attention
def delta_rule(K, V, beta, state=None):
    """The delta rule: a fixed-size memory that is written to, not appended to.

    Softmax attention keeps every key and value and re-reads them all at every step.
    A delta-rule layer keeps one d x d matrix S and updates it in place:

        S <- S (I - beta_t k_t k_t^T) + beta_t v_t k_t^T

    The first term erases whatever S currently associates with k_t, the second writes
    v_t there instead - which is why it is called the delta rule and not just a sum.
    A real layer reads with its own query projection, o_t = S_t q_t; this function
    probes with the key itself, because what it is here to measure is what was stored.

    Gated variants add a forget factor so old associations decay instead of persisting
    until overwritten: Gated DeltaNet uses one scalar per head, and Kimi's KDA makes it
    per channel (a diagonal gate).  The shape of the state, and the fact that it does
    not grow with the sequence, are the same.
    """
    n, d = K.shape
    dv = V.shape[1]
    S = np.zeros((dv, d)) if state is None else state
    out = np.zeros((n, dv))
    for t in range(n):
        k = K[t] / (np.linalg.norm(K[t]) + 1e-12)
        b = beta[t]
        out[t] = S @ k
        S = S - b * np.outer(S @ k, k) + b * np.outer(V[t], k)
    return out, S


def delta_rule_recall(d=16, n=8, seed=0, orthogonal=True):
    """Write n key-value pairs into one d x d state, then read all n of them back.

    With orthogonal keys the write of one pair does not disturb the others, so the
    state holds exactly d associations perfectly.  Ask for the d+1-th and something
    has to be overwritten.  With ordinary random keys nothing is orthogonal and every
    write smears the ones already there, so the error starts above zero immediately.
    """
    rng = np.random.default_rng(seed)
    if orthogonal:
        basis = np.linalg.qr(rng.standard_normal((d, d)))[0]
        K = np.array([basis[t % d] for t in range(n)])
    else:
        K = rng.standard_normal((n, d))
    V = rng.standard_normal((n, d))
    _, S = delta_rule(K, V, np.ones(n))
    errs = []
    for t in range(n):
        k = K[t] / np.linalg.norm(K[t])
        errs.append(float(np.linalg.norm(S @ k - V[t]) / np.linalg.norm(V[t])))
    return np.array(errs)


print(f'{"pairs":>7}{"worst recall error":>22}{"associations lost":>20}')
for n in (8, 16, 17, 24, 32):
    e = delta_rule_recall(d=16, n=n)
    print(f'{n:>7}{e.max():>22.2e}{int((e > 0.5).sum()):>20}')
print()
print('with ordinary random keys instead of orthogonal ones, 8 pairs already give',
      f'{delta_rule_recall(d=16, n=8, orthogonal=False).max():.2f}')

  pairs    worst recall error   associations lost
      8              1.00e-12                   0
     16              1.00e-12                   0
     17              1.33e+00                   1
     24              1.69e+00                   8
     32              1.87e+00                  16

with ordinary random keys instead of orthogonal ones, 8 pairs already give 1.66


## LeetCode 239 — Sliding Window Maximum

A deque of indices whose values are strictly decreasing. A value that arrives kills
everything smaller behind it, because the newcomer is both larger *and* newer. This is
the running maximum online softmax carries.

In [9]:
# ================================================================ 7. LeetCode 239
class SolutionSlidingWindowMaximum:
    """LeetCode 239 - Sliding Window Maximum.

    A deque holding indices whose values are strictly decreasing.  The front is the
    maximum of the current window; anything smaller than the value arriving is dead,
    because the newcomer is both larger and newer, so it outlives them all.

    Online softmax carries the same kind of running maximum - one number instead of
    the whole row - and rescales everything already accumulated whenever it changes.
    """

    def maxSlidingWindow(self, nums: list[int], k: int) -> list[int]:
        from collections import deque
        dq, out = deque(), []
        for i, v in enumerate(nums):
            while dq and nums[dq[-1]] <= v:
                dq.pop()                      # smaller AND older: never the max again
            dq.append(i)
            if dq[0] <= i - k:
                dq.popleft()                  # the front fell out of the window
            if i >= k - 1:
                out.append(nums[dq[0]])
        return out


def max_sliding_window_brute(nums, k):
    return [max(nums[i:i + k]) for i in range(len(nums) - k + 1)]


nums, k = [1, 3, -1, -3, 5, 3, 6, 7], 3
print(nums, 'k =', k, '->', SolutionSlidingWindowMaximum().maxSlidingWindow(nums, k))
print('brute force            ->', max_sliding_window_brute(nums, k))

[1, 3, -1, -3, 5, 3, 6, 7] k = 3 -> [3, 3, 5, 5, 6, 7]
brute force            -> [3, 3, 5, 5, 6, 7]


## LeetCode 703 — Kth Largest Element in a Stream

A min-heap of exactly `k` elements: the root is the k-th largest so far, and the memory
is `k` rather than the length of the stream. One pass over something you cannot store,
and a bounded amount of state that has to be enough.

In [10]:
# ================================================================ 8. LeetCode 703
class KthLargest:
    """LeetCode 703 - Kth Largest Element in a Stream.

    Keep a min-heap of exactly k elements.  Its root is the k-th largest seen so far,
    and everything smaller is discarded the moment it arrives, so the memory is k and
    not the length of the stream.

    That is the shape FlashAttention and linear attention both have: a stream you only
    get one pass over, and a bounded amount of state that has to be enough.
    """

    def __init__(self, k: int, nums: list[int]):
        import heapq
        self.k = k
        self.heap = list(nums)
        heapq.heapify(self.heap)
        while len(self.heap) > k:
            heapq.heappop(self.heap)

    def add(self, val: int) -> int:
        import heapq
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, val)
        elif val > self.heap[0]:
            heapq.heapreplace(self.heap, val)
        return self.heap[0]


def kth_largest_brute(k, nums, adds):
    seen, out = list(nums), []
    for v in adds:
        seen.append(v)
        out.append(sorted(seen, reverse=True)[k - 1])
    return out


kl = KthLargest(3, [4, 5, 8, 2])
print([kl.add(v) for v in [3, 5, 10, 9, 4]])

[4, 5, 5, 8, 8]


## Tests

Everything above, re-derived from scratch and asserted.

In [11]:
# ================================================================ 10. assertions
def check():
    rng = np.random.default_rng(0)

    # softmax: rows sum to 1, and the max trick changes nothing but the overflow
    for _ in range(50):
        x = rng.standard_normal((4, 7)) * 3
        assert np.allclose(softmax(x).sum(axis=-1), 1.0)
        assert np.allclose(softmax(x), softmax_naive(x), atol=1e-12)
        # shifting the input by any constant leaves softmax unchanged
        assert np.allclose(softmax(x), softmax(x + 1000.0), atol=1e-12)
    with np.errstate(over='ignore', invalid='ignore'):
        big = np.array([[1000.0, 1001.0, 999.0]])
        assert np.isnan(softmax_naive(big)).any()      # the naive one dies
    assert np.isfinite(softmax(big)).all()             # the shifted one does not

    # attention: weights are a distribution, output is inside the convex hull of V
    for _ in range(30):
        n, m, d, dv = 5, 9, 8, 4
        Q, K, V = (rng.standard_normal((n, d)), rng.standard_normal((m, d)),
                   rng.standard_normal((m, dv)))
        out, w = attention(Q, K, V)
        assert np.allclose(w.sum(axis=-1), 1.0) and (w >= 0).all()
        assert (out.max(axis=0) <= V.max(axis=0) + 1e-9).all()
        assert (out.min(axis=0) >= V.min(axis=0) - 1e-9).all()
        # one pass in blocks gives the same number, not an approximation
        for blk in (1, 2, 3, 4, 16):
            assert np.allclose(out, attention_online(Q, K, V, block=blk), atol=1e-12)
        msk = causal_mask(n, m)
        out_c, w_c = attention(Q, K, V, mask=msk)
        assert np.allclose(w_c.sum(axis=-1), 1.0)
        assert (w_c[~msk] == 0).all()                  # nothing leaks from the future
        for blk in (1, 3, 16):
            assert np.allclose(out_c, attention_online(Q, K, V, block=blk, mask=msk),
                               atol=1e-12)
    # masking after the softmax leaves rows that do not sum to 1
    msk = causal_mask(5, 5)
    Q, K, V = rng.standard_normal((5, 8)), rng.standard_normal((5, 8)), rng.standard_normal((5, 3))
    _, w_bad = attention_mask_after_softmax(Q, K, V, msk)
    assert not np.allclose(w_bad.sum(axis=-1), 1.0)
    assert abs(w_bad.sum(axis=-1)[0]) < 1.0            # the first row loses the most

    # scaling: without 1/sqrt(d) the logits grow with d and the softmax saturates
    sharp = []
    for d in (8, 64, 512):
        q, k = rng.standard_normal((200, d)), rng.standard_normal((200, d))
        sharp.append(float(np.std(q @ k.T)))
    assert sharp[0] < sharp[1] < sharp[2]              # std grows like sqrt(d)

    # KV cache arithmetic
    mha = kv_cache_bytes_per_token()
    mqa = kv_cache_bytes_per_token(kv_heads=1)
    gqa = kv_cache_bytes_per_token(kv_heads=8)
    mla = kv_cache_bytes_per_token(latent=512, rope_dim=64)
    assert mha == 32 * 2 * 32 * 128 * 2 == 524288
    assert mqa * 32 == mha and gqa * 4 == mha
    assert mla == 32 * (512 + 64) * 2 == 36864
    assert mqa < mla < gqa < mha
    # a linear-attention state does not depend on how long the sequence is
    assert linear_state_bytes() == 32 * 32 * 128 * 128 * 2

    # paging: never more than page_size - 1 wasted slots per sequence
    lens = [1, 15, 16, 17, 300, 4096]
    for page in (8, 16, 64):
        pages, reserved, used, _ = paged_waste(lens, page)
        assert reserved >= used and reserved - used <= len(lens) * (page - 1)

    # the delta rule writes in place: d orthogonal associations fit exactly
    assert delta_rule_recall(d=16, n=16).max() < 1e-9
    over = delta_rule_recall(d=16, n=17)               # one more than the state holds
    assert over.max() > 0.5 and over[-1] < 1e-9        # the newest is perfect, one is gone
    smear = delta_rule_recall(d=16, n=8, orthogonal=False)
    assert smear.max() > 1e-3                          # non-orthogonal keys interfere

    # LeetCode 239 against brute force
    for seed in range(300):
        r = np.random.default_rng(seed)
        nums = r.integers(-50, 50, int(r.integers(1, 20))).tolist()
        k = int(r.integers(1, len(nums) + 1))
        assert SolutionSlidingWindowMaximum().maxSlidingWindow(nums, k) == \
            max_sliding_window_brute(nums, k)
    assert SolutionSlidingWindowMaximum().maxSlidingWindow([1, 3, -1, -3, 5, 3, 6, 7], 3) \
        == [3, 3, 5, 5, 6, 7]
    assert SolutionSlidingWindowMaximum().maxSlidingWindow([1], 1) == [1]

    # LeetCode 703 against brute force
    for seed in range(300):
        r = np.random.default_rng(seed + 99)
        k = int(r.integers(1, 6))
        init = r.integers(-20, 20, int(r.integers(0, 6))).tolist()
        adds = r.integers(-20, 20, int(r.integers(1, 8))).tolist()
        if len(init) + 1 < k:
            continue                                   # 703 guarantees k-th exists
        kl = KthLargest(k, list(init))
        got = [kl.add(v) for v in adds]
        assert got == kth_largest_brute(k, init, adds)
    kl = KthLargest(3, [4, 5, 8, 2])
    assert [kl.add(v) for v in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
    return True


print('check() ->', check())

check() -> True
